## 1. Load + confirm:

In [8]:
import os
os.environ['KMP_DUPLICATE_LIB_OK']='True'

import pandas as pd
import numpy as np

pd.set_option('display.max_columns',None)

from features import (
    get_team_form_fast, get_attack_defense_fast,
    get_clean_sheet_rate, get_days_rest,
    get_second_half_goals_pct, get_penalty_goal_pct,
    get_h2h_fast, calculate_elo_history, get_elo_at_date,
    build_shootout_index, get_shootout_win_rate,
    calculate_lambda, build_score_matrix,build_team_match_index,
    build_goals_index    
)


In [15]:
import pandas as pd
import numpy as np

# ---- Load ----
results_new  = pd.read_csv('../data/raw/results_new.csv', parse_dates=['date'])
former_names = pd.read_csv('../data/raw/former_names.csv', parse_dates=['start_date','end_date'])
goals_df     = pd.read_csv('../data/raw/goalscorers.csv', parse_dates=['date'])
shootouts_df = pd.read_csv('../data/raw/shootouts.csv', parse_dates=['date'])

print(f"Raw results_new.csv : {results_new.shape}")

# ---- Step 1: Set aside unplayed fixtures (null scores) -- these are what we'll predict later ----
upcoming_fixtures = results_new[results_new['home_score'].isnull()].copy()
results_new = results_new[results_new['home_score'].notnull()].copy()
print(f"\nUpcoming (unplayed) fixtures set aside: {len(upcoming_fixtures)}")
print(upcoming_fixtures[['date','home_team','away_team','tournament']].to_string(index=False))

# ---- Step 2: Deduplicate ----
before = len(results_new)
results_new = results_new.drop_duplicates(
    subset=['date','home_team','away_team','home_score','away_score'], keep='first')
print(f"\nDropped {before - len(results_new)} exact duplicate row(s)")

dupe_keys = results_new.duplicated(subset=['date','home_team','away_team'], keep=False)
conflicting = results_new[dupe_keys]
if len(conflicting) > 0:
    print(f"Dropping {len(conflicting)} row(s) with conflicting scores for the same fixture (unverifiable):")
    print(conflicting[['date','home_team','away_team','home_score','away_score']].to_string(index=False))
    results_new = results_new[~dupe_keys]

# ---- Step 3: Former-name remapping, point-in-time correct, applied consistently everywhere ----
def build_remap_fn(df):
    def remap(row_date, team_name):
        m = df[(df['former'] == team_name) &
               ((df['start_date'].isnull()) | (df['start_date'] <= row_date)) &
               ((df['end_date'].isnull())   | (df['end_date']   >= row_date))]
        return m.iloc[0]['current'] if len(m) > 0 else team_name
    return remap

remap = build_remap_fn(former_names)

for col in ['home_team', 'away_team']:
    results_new[col] = results_new.apply(lambda r: remap(r['date'], r[col]), axis=1)
for col in ['home_team', 'away_team', 'team']:
    goals_df[col] = goals_df.apply(lambda r: remap(r['date'], r[col]), axis=1)
for col in ['home_team', 'away_team', 'winner', 'first_shooter']:
    shootouts_df[col] = shootouts_df.apply(lambda r: remap(r['date'], r[col]), axis=1)

print("\n✅ Former-name remapping applied to results, goalscorers, and shootouts")

# ---- Step 4: Post-2000 filter ----
results_2000 = results_new[results_new['date'].dt.year >= 2000].copy()

# ---- Step 5: Tournament keep-list filter (established earlier) ----
tournaments_to_keep = [
    'FIFA World Cup', 'FIFA World Cup qualification',
    'UEFA Euro', 'UEFA Euro qualification', 'UEFA Nations League',
    'African Cup of Nations', 'African Cup of Nations qualification',
    'AFC Asian Cup', 'AFC Asian Cup qualification',
    'Copa América', 'Gold Cup',
    'CONCACAF Nations League', 'CONCACAF Nations League qualification',
    'Confederations Cup', 'Oceania Nations Cup',
    'WAFF Championship', 'Arab Cup', 'CAFA Nations Cup', 'Amílcar Cabral Cup'
]

clean_df = results_2000[results_2000['tournament'].isin(tournaments_to_keep)].copy()
clean_df = clean_df.dropna(subset=['home_score','away_score']).reset_index(drop=True)

print(f"\nFinal clean_matches shape : {clean_df.shape}")
print(f"Tournaments kept          : {clean_df['tournament'].nunique()}")
print(f"Date range                : {clean_df['date'].min()} to {clean_df['date'].max()}")

# ---- Save everything ----
clean_df.to_csv('../data/processed/clean_matches.csv', index=False)
upcoming_fixtures.to_csv('../data/wc2026/upcoming_fixtures.csv', index=False)
goals_df.to_csv('../data/raw/goalscorers.csv', index=False)
shootouts_df.to_csv('../data/raw/shootouts.csv', index=False)

print("\n✅ Saved: clean_matches.csv, upcoming_fixtures.csv (overwritten goalscorers.csv/shootouts.csv with remapped names)")

Raw results_new.csv : (49501, 9)

Upcoming (unplayed) fixtures set aside: 0
Empty DataFrame
Columns: [date, home_team, away_team, tournament]
Index: []

Dropped 1 exact duplicate row(s)
Dropping 2 row(s) with conflicting scores for the same fixture (unverifiable):
      date home_team     away_team  home_score  away_score
1974-02-17    Tahiti New Caledonia           2           1
1974-02-17    Tahiti New Caledonia           1           2

✅ Former-name remapping applied to results, goalscorers, and shootouts

Final clean_matches shape : (13052, 9)
Tournaments kept          : 19
Date range                : 2000-01-22 00:00:00 to 2026-07-07 00:00:00

✅ Saved: clean_matches.csv, upcoming_fixtures.csv (overwritten goalscorers.csv/shootouts.csv with remapped names)


## 2. build all indices + elo history:

In [16]:
import importlib
import features
importlib.reload(features)
from features import (
    build_team_match_index, build_goals_index, build_shootout_index,
    calculate_elo_history
)

clean_df    = pd.read_csv('../data/processed/clean_matches.csv', parse_dates=['date'])
goals_df    = pd.read_csv('../data/raw/goalscorers.csv', parse_dates=['date'])
shootouts_df = pd.read_csv('../data/raw/shootouts.csv', parse_dates=['date'])

team_index     = build_team_match_index(clean_df)
goals_index    = build_goals_index(goals_df)
shootout_index = build_shootout_index(shootouts_df)
elo_history, final_elo = calculate_elo_history(clean_df)

print(f"team_index      : {len(team_index)} teams")
print(f"goals_index     : {len(goals_index)} teams")
print(f"shootout_index  : {len(shootout_index)} teams")
print(f"elo_history     : {len(elo_history)} rows")

# Sanity checks
print("\n'Netherlands Antilles' still present (should be False/absent):", 'Netherlands Antilles' in team_index)
print("'Curaçao' present (should be True):", 'Curaçao' in team_index or 'Curacao' in team_index)

# Confirm goals_for/goals_against fix is intact on the fresh data
sample = list(team_index.keys())[0]
print(f"\n{sample} columns:", list(team_index[sample].columns))

team_index      : 217 teams
goals_index     : 220 teams
shootout_index  : 230 teams
elo_history     : 26104 rows

'Netherlands Antilles' still present (should be False/absent): False
'Curaçao' present (should be True): True

Ghana columns: ['date', 'home_team', 'away_team', 'home_score', 'away_score', 'tournament', 'city', 'country', 'neutral']


In [17]:
import importlib
import features
importlib.reload(features)
from features import build_team_match_index

team_index = build_team_match_index(clean_df)
print(team_index['Ghana'][['date','home_team','away_team','home_score','away_score','goals_for','goals_against']].head())

        date home_team     away_team  home_score  away_score  goals_for  \
0 2000-01-22     Ghana      Cameroon           1           1          1   
1 2000-01-27     Ghana          Togo           2           0          2   
2 2000-01-31     Ghana   Ivory Coast           0           2          0   
3 2000-02-06     Ghana  South Africa           0           1          0   
4 2000-04-08  Tanzania         Ghana           0           1          1   

   goals_against  
0              1  
1              0  
2              2  
3              1  
4              0  


## 3. full feature matrix:

In [20]:
from tqdm import tqdm

# Rebuild everything fresh from the corrected clean_matches.csv / goalscorers.csv / shootouts.csv
team_index     = build_team_match_index(clean_df)
goals_index    = build_goals_index(goals_df)
shootout_index = build_shootout_index(shootouts_df)
elo_history, final_elo = calculate_elo_history(clean_df)

avg_home_goals = clean_df['home_score'].mean()
avg_away_goals = clean_df['away_score'].mean()
avg_goals = (avg_home_goals + avg_away_goals) / 2
print(f"avg_goals: {avg_goals:.4f}")

# Tournament weights — hand-crafted, domain-based, one entry per kept tournament (19)
tournament_weights = {
    'FIFA World Cup'                        : 1.00,
    'UEFA Euro'                             : 0.85,
    'Copa América'                          : 0.85,
    'African Cup of Nations'                : 0.80,
    'AFC Asian Cup'                         : 0.80,
    'Gold Cup'                              : 0.75,
    'CONCACAF Nations League'               : 0.75,
    'UEFA Nations League'                   : 0.70,
    'Confederations Cup'                    : 0.75,
    'Oceania Nations Cup'                   : 0.70,
    'FIFA World Cup qualification'          : 0.65,
    'UEFA Euro qualification'               : 0.65,
    'African Cup of Nations qualification'  : 0.60,
    'AFC Asian Cup qualification'           : 0.60,
    'CONCACAF Nations League qualification' : 0.55,
    'WAFF Championship'                     : 0.50,
    'Arab Cup'                              : 0.50,
    'Amílcar Cabral Cup'                    : 0.45,
    'CAFA Nations Cup'                      : 0.45,
}

# Recency weight — continuous exponential decay by date (agreed earlier, replacing the hard bucket-split idea)
# Half-life of 4 years: a match is half as influential every 4 years in the past, relative to the most recent match
max_date = clean_df['date'].max()
HALFLIFE_DAYS = 365 * 4

rows = []
for _, match in tqdm(clean_df.iterrows(), total=len(clean_df), desc="Building V2 feature matrix"):
    h, a, d = match['home_team'], match['away_team'], match['date']

    home_form = get_team_form_fast(h, d, team_index)
    away_form = get_team_form_fast(a, d, team_index)
    home_attack, home_defense = get_attack_defense_fast(h, d, team_index)
    away_attack, away_defense = get_attack_defense_fast(a, d, team_index)
    home_elo = get_elo_at_date(h, d, elo_history)
    away_elo = get_elo_at_date(a, d, elo_history)

    home_form  = home_form  if home_form  is not None else np.nan
    away_form  = away_form  if away_form  is not None else np.nan
    home_attack  = home_attack  if home_attack  is not None else np.nan
    home_defense = home_defense if home_defense is not None else np.nan
    away_attack  = away_attack  if away_attack  is not None else np.nan
    away_defense = away_defense if away_defense is not None else np.nan

    h2h = get_h2h_fast(h, a, d, team_index)
    h2h_matches = h2h['matches']
    h2h_home_win_rate = h2h['a_win_rate'] if h2h['a_win_rate'] is not None else 0.5

    elo_diff = home_elo - away_elo
    elo_closeness = 1 / (1 + abs(elo_diff))
    defensive_strength = min(1 / (home_defense + away_defense + 0.1), 3.0)  # capped, per earlier fix

    row = {
        'home_form': home_form, 'home_attack': home_attack, 'home_defense': home_defense, 'home_elo': home_elo,
        'away_form': away_form, 'away_attack': away_attack, 'away_defense': away_defense, 'away_elo': away_elo,
        'h2h_matches': h2h_matches, 'h2h_home_win_rate': h2h_home_win_rate,
        'elo_diff': elo_diff, 'elo_closeness': elo_closeness, 'defensive_strength': defensive_strength,
        'home_clean_sheet_rate': get_clean_sheet_rate(h, d, team_index),
        'away_clean_sheet_rate': get_clean_sheet_rate(a, d, team_index),
        'home_days_rest': get_days_rest(h, d, team_index),
        'away_days_rest': get_days_rest(a, d, team_index),
        'home_second_half_goals_pct': get_second_half_goals_pct(h, d, goals_index),
        'away_second_half_goals_pct': get_second_half_goals_pct(a, d, goals_index),
        'home_penalty_goal_pct': get_penalty_goal_pct(h, d, goals_index),
        'away_penalty_goal_pct': get_penalty_goal_pct(a, d, goals_index),
        'home_shootout_win_rate': get_shootout_win_rate(h, d, shootout_index),
        'away_shootout_win_rate': get_shootout_win_rate(a, d, shootout_index),
        'home_team': h, 'away_team': a, 'date': d,
        'tournament': match['tournament'],
        'tournament_weight': tournament_weights.get(match['tournament']),
        'recency_weight': 0.5 ** ((max_date - d).days / HALFLIFE_DAYS),
    }

    if match['home_score'] > match['away_score']:
        row['result'] = 2
    elif match['home_score'] == match['away_score']:
        row['result'] = 1
    else:
        row['result'] = 0

    row['home_score'] = match['home_score']
    row['away_score'] = match['away_score']

    rows.append(row)

feature_df = pd.DataFrame(rows)

# Cold-start handling (Option C, as decided earlier): flag first, then fill with dataset average
cold_start_cols = ['home_form','away_form','home_attack','home_defense','away_attack','away_defense']
print("Cold-start nulls per column (before fill):")
print(feature_df[cold_start_cols].isnull().sum())

feature_df['home_has_full_history'] = feature_df['home_form'].notnull().astype(int)
feature_df['away_has_full_history'] = feature_df['away_form'].notnull().astype(int)

for col in cold_start_cols:
    feature_df[col] = feature_df[col].fillna(feature_df[col].mean())

# Recompute defensive_strength now that defense values are filled, then apply the cap
feature_df['defensive_strength'] = (1 / (feature_df['home_defense'] + feature_df['away_defense'] + 0.1)).clip(upper=3.0)

print(f"\nRemaining nulls anywhere: {feature_df.isnull().sum().sum()}")

print(f"\nShape: {feature_df.shape}")
print(f"Null tournament_weight: {feature_df['tournament_weight'].isnull().sum()}")
print(f"Total nulls:\n{feature_df.isnull().sum()[feature_df.isnull().sum() > 0]}")
print(f"\nResult distribution:\n{feature_df['result'].value_counts(normalize=True).round(3)}")
print(f"\nrecency_weight range: {feature_df['recency_weight'].min():.4f} to {feature_df['recency_weight'].max():.4f}")

avg_goals: 1.3761


Building V2 feature matrix: 100%|██████████| 13052/13052 [01:08<00:00, 191.25it/s]

Cold-start nulls per column (before fill):
home_form       107
away_form       110
home_attack     107
home_defense    107
away_attack     110
away_defense    110
dtype: int64

Remaining nulls anywhere: 0

Shape: (13052, 34)
Null tournament_weight: 0
Total nulls:
Series([], dtype: int64)

Result distribution:
result
2    0.486
0    0.296
1    0.218
Name: proportion, dtype: float64

recency_weight range: 0.0102 to 1.0000


## 4. Reorder columns + save

In [21]:
cols = [
    # Features (25)
    'home_form', 'home_attack', 'home_defense', 'home_elo', 'home_has_full_history',
    'away_form', 'away_attack', 'away_defense', 'away_elo', 'away_has_full_history',
    'h2h_matches', 'h2h_home_win_rate', 'elo_diff', 'elo_closeness', 'defensive_strength',
    'home_clean_sheet_rate', 'away_clean_sheet_rate',
    'home_days_rest', 'away_days_rest',
    'home_second_half_goals_pct', 'away_second_half_goals_pct',
    'home_penalty_goal_pct', 'away_penalty_goal_pct',
    'home_shootout_win_rate', 'away_shootout_win_rate',
    # Metadata (6)
    'home_team', 'away_team', 'date', 'tournament', 'tournament_weight', 'recency_weight',
    # Targets (3)
    'result', 'home_score', 'away_score'
]

feature_df = feature_df[cols]
feature_df.to_csv('../data/processed/feature_matrix.csv', index=False)

print(f"Saved — shape: {feature_df.shape}")
print(f"\nRange sanity checks:")
print(f"  elo_closeness       : {feature_df['elo_closeness'].min():.4f} to {feature_df['elo_closeness'].max():.4f}  (expect 0-1)")
print(f"  defensive_strength  : {feature_df['defensive_strength'].min():.4f} to {feature_df['defensive_strength'].max():.4f}  (expect capped at 3.0)")
print(f"  tournament_weight   : {feature_df['tournament_weight'].min():.4f} to {feature_df['tournament_weight'].max():.4f}  (expect 0.45-1.0)")
print(f"  h2h_home_win_rate   : {feature_df['h2h_home_win_rate'].min():.4f} to {feature_df['h2h_home_win_rate'].max():.4f}  (expect 0-1)")

Saved — shape: (13052, 34)

Range sanity checks:
  elo_closeness       : 0.0012 to 1.0000  (expect 0-1)
  defensive_strength  : 0.0352 to 3.0000  (expect capped at 3.0)
  tournament_weight   : 0.4500 to 1.0000  (expect 0.45-1.0)
  h2h_home_win_rate   : 0.0000 to 1.0000  (expect 0-1)
